In [13]:
import pandas as pd
from pathlib import Path

In [14]:
base_dir = Path("/mnt/c/Users/jonan/Documents/1Work/RoseLab/Spatial/CAR_T/Results/spp1_mac_analysis/lr_integration/lr_levels/high/")
de_file = base_dir / "high__M2_vs_M1__DE.csv"
de = pd.read_csv(de_file)

In [15]:
de['log2FC'].describe()

count    11449.000000
mean         0.472987
std          1.632823
min        -25.549032
25%          0.268747
50%          0.597947
75%          0.882011
max         25.669664
Name: log2FC, dtype: float64

In [16]:
de = de[(de['log2FC'] < 20) & (de['log2FC'] > -20)]

In [17]:
de['log2FC'].describe()

count    11407.000000
mean         0.547523
std          0.607284
min         -2.959437
25%          0.272208
50%          0.599667
75%          0.882447
max          5.133067
Name: log2FC, dtype: float64

In [10]:
de.columns

Index(['gene', 'log2FC', 'pval', 'padj', 'scores'], dtype='object')

In [24]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- inputs ----
# assumes you already have a DataFrame `de` with columns:
# ['gene', 'log2FC', 'pval', 'padj', 'scores']
base_dir = Path("/mnt/c/Users/jonan/Documents/1Work/RoseLab/Spatial/CAR_T/Results/spp1_mac_analysis/lr_integration/lr_levels/high/")
png_path = base_dir / "volcano.png"
pdf_path = base_dir / "volcano.pdf"

# thresholds (tune as needed)
lfc_thresh = 1.0            # absolute log2FC threshold
fdr_thresh = 0.05           # significance threshold

# ---- prep ----
de = de.copy()

# choose adjusted p if present & non-null; otherwise fall back to pval
p_col = "padj" if ("padj" in de.columns and de["padj"].notna().any()) else "pval"

# replace 0 or negative p-values (if any) with the smallest positive float to avoid -log10(0)
pvals = de[p_col].astype(float).values
pvals[pvals <= 0] = np.nextafter(0, 1)
neglog10p = -np.log10(pvals)

# classification
is_up = (de["log2FC"] >= lfc_thresh) & (de[p_col] < fdr_thresh)
is_dn = (de["log2FC"] <= -lfc_thresh) & (de[p_col] < fdr_thresh)
is_sig = is_up | is_dn
is_ns = ~is_sig

# ---- plot ----
plt.figure(figsize=(7, 6), dpi=150)

# non-sig first (so sig points draw on top) — NS grey, sig all same color
plt.scatter(de.loc[is_ns, "log2FC"], neglog10p[is_ns], s=8, alpha=0.5, color="#bfbfbf")
plt.scatter(de.loc[is_sig, "log2FC"], neglog10p[is_sig], s=12, alpha=0.9)  # default color for all sig

# thresholds
plt.axvline(x= lfc_thresh, linestyle="--", linewidth=1, color="#666666")
plt.axvline(x=-lfc_thresh, linestyle="--", linewidth=1, color="#666666")
plt.axhline(y=-np.log10(fdr_thresh), linestyle="--", linewidth=1, color="#666666")

plt.xlabel("log2 fold-change")
plt.ylabel(f"-log10({p_col})")
plt.title("M2 vs M1 - High Spp1-CD44 interactions")

# no legend per request

# ---- annotate Tgfb1 ----
gene_name = "Tgfb1"

# prefer exact match; if multiple, pick most significant (max -log10 p)
exact = de["gene"] == gene_name
ci = de["gene"].str.lower() == gene_name.lower()

if exact.any() or ci.any():
    rows = de.loc[exact] if exact.any() else de.loc[ci]
    # choose the row with smallest p (largest -log10 p)
    idx = rows.index[np.argmax(neglog10p[rows.index])]
    x = float(de.loc[idx, "log2FC"])
    y = float(neglog10p[idx])
    # highlight point
    plt.scatter([x], [y], s=50, edgecolor="black", facecolor="yellow", zorder=5)
    # label with an arrow
    plt.annotate(
        gene_name,
        xy=(x, y),
        xytext=(x + (0.6 if x >= 0 else -0.6), y + 0.6),
        arrowprops=dict(arrowstyle="->", lw=1),
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85),
    )
else:
    print(f"Warning: {gene_name} not found in `de['gene']` — skipping annotation.")

plt.tight_layout()

# ---- save ----
base_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(png_path, dpi=300)
plt.savefig(pdf_path)
plt.close()

print(f"Saved: {png_path}")
print(f"Saved: {pdf_path}")


Saved: /mnt/c/Users/jonan/Documents/1Work/RoseLab/Spatial/CAR_T/Results/spp1_mac_analysis/lr_integration/lr_levels/high/volcano.png
Saved: /mnt/c/Users/jonan/Documents/1Work/RoseLab/Spatial/CAR_T/Results/spp1_mac_analysis/lr_integration/lr_levels/high/volcano.pdf
